# Downsampling approaches
Exploring & implementing the different approaches to downsampling a genome

## MinHashing
Confirmed downsampling approach from Special Course.

Pros:
- Fast & efficient

Cons:
- Hashing algorithm unavailable; backtracking hash values to kmers is not possible (e.g. feature attributions for NNs)

### Constructing singular signatures

In [ ]:
import os, sys 
from manipulations import construct_SM_sketches
from io_operations import presence_matrix
from paths import raw_data_path, data_prod_path

### Phage Minhash Sketch Construction ###
# pk = 12
# pn = 500
# phage_outdir = f"PhageMinhash_n{pn}_k{pk}/"
# construct_SM_sketches(raw_in = raw_data_path+"phagehost_KU/phage_cleaned.fasta", 
#                       k = pk, 
#                       outdir = phage_outdir, 
#                       quiet = False,
#                       sourmash_parameters=[pn, 0])

### Bacteria Minhash Sketch Construction ###
bk = 16
bn = 500
bact_outdir = f"BactMinhash_n{bn}_k{bk}/"
construct_SM_sketches(raw_in = raw_data_path+"phagehost_KU/bacteria_fasta/", 
                      k = bk, 
                      outdir = bact_outdir, 
                      quiet = False,
                      sourmash_parameters=[bn, 0])


In [ ]:
from pickle import dump
### Presence Matrix
presence_outdir = f"SM_sketches/PresMat_bn{bn}_bk{bk}_pn{pn}_pk{pk}/"
try:
    os.makedirs(data_prod_path+presence_outdir)
except FileExistsError: # directory already exists
    pass

try:
    binary_matrix, entity_to_index, minhash_to_index, phage_minhash_data, bact_minhash_data = presence_matrix(
        phage_minhash_dir=data_prod_path+"SM_sketches/"+phage_outdir, 
        bact_minhash_dir=data_prod_path+"SM_sketches/"+bact_outdir,
        k=[bk, pk],
        n=[bn, pn],
        reversecomp_data=False, TS=True)
except Exception as e:
    raise RuntimeError(f"Failed to run presence_matrix succesfully, exception: {e}")

try:
    with open(data_prod_path+presence_outdir+"binary_matrix.pkl", "wb") as binary_matrix_file:
        dump(binary_matrix, binary_matrix_file)
    with open(data_prod_path+presence_outdir+"entity_to_index.pkl", "wb") as entity_to_index_file:
        dump(entity_to_index, entity_to_index_file)
    with open(data_prod_path+presence_outdir+"minhash_to_index.pkl", "wb") as minhash_to_index_file:
        dump(minhash_to_index, minhash_to_index_file)
    with open(data_prod_path+presence_outdir+"phage_minhash_data.pkl", "wb") as phage_minhash_data_file:
        dump(phage_minhash_data, phage_minhash_data_file)
    with open(data_prod_path+presence_outdir+"bact_minhash_data.pkl", "wb") as bact_minhash_data_file:
        dump(bact_minhash_data, bact_minhash_data_file)
except Exception as e:
    print(f"Failed to save presence_matrix results to {presence_outdir}:\n{e}")

### Constructing multiple signatures

In [ ]:
from manipulations import construct_SM_sketches
import os, sys 
raw_data_path = "../raw_data/"
data_prod_path = "../data_prod/"

for k in [6, 9, 12, 15, 18, 24]:
    for n in [50, 100, 500, 1000, 5000]:
        #Phages minhash sketches
        construct_SM_sketches(fasta = raw_data_path+"phagehost_KU/phage_cleaned.fasta", 
                            k = k, 
                            outdir = f"PhageMinhash_n{n}_k{k}_rev/", 
                            quiet = False,
                            sourmash_parameters=[n, 0],
                            include_reverse=True)
        
        #Bacteria minhash sketches
        construct_SM_sketches(fasta = raw_data_path+"phagehost_KU/bacteriaKU_cleaned.fasta", 
                        k = k, 
                        outdir = f"BactMinhash_n{n}_k{k}_rev/", 
                        quiet = False,
                        sourmash_parameters=[n, 0],
                        include_reverse=True)

### Behind the curtains 
for constructing the functions construct_SM_sketches

In [ ]:
import sourmash
import pandas as pd
from Bio import SeqIO
import os, sys
from tqdm import tqdm
import numpy as np

def construct_SM_sketches_dev(raw_in : str, k : int, outdir : str, quiet : bool = False, sourmash_parameters = [50000, 0], include_reverse : bool = False) -> int:
    """
    Construct sourmash sketches given a fasta input.
    
    Args:
        *raw_in* (str): Path to the input fasta file or directory containing fasta files.
        *k* (int): Length of the k-mers. 
        *outdir* (str): directory for storing sketches (each signature in its own file)
        *quiet* (bool): If True, suppress progress output. Default is False.
        *sourmash_parameters* (list): specify sourmash.MinHash(n, scaled)
        *include_reverse* (bool): include the reverse strand to sketches
    
    Returns:
        *exit_status* (binary): 0 for success, 1 for failure.
    """
    import sourmash

    ### Input Control ###
    if type(outdir) is not str:
        raise ValueError("outdir must be a path")
    
    if not os.path.exists(data_prod_path+"SM_sketches/"):
        try:
            os.makedirs(data_prod_path+"SM_sketches/", exist_ok=True)
            if not quiet: print(f"Created output directory: {data_prod_path}SM_sketches/")
        except OSError as e:
            raise ValueError(f"Could not create outdir {data_prod_path}SM_sketches/: {e}")
    
    # Ensure outdir exists (create if missing)
    if not os.path.exists(data_prod_path+"SM_sketches/"+outdir):
        try:
            os.makedirs(data_prod_path+"SM_sketches/"+outdir, exist_ok=True)
            if not quiet: print(f"Created output directory: {outdir}")
        except OSError as e:
            raise ValueError(f"Could not create outdir {outdir}: {e}")
    elif not os.path.isdir(data_prod_path+"SM_sketches/"+outdir):
        raise ValueError(f"outdir exists but is not a directory: {outdir}")

    outpath = data_prod_path+"SM_sketches/"+outdir
    if not quiet: print(f"Output path for sketches: {outpath}")

    # Ensuring sourmash parameters are appropriate
    if sourmash_parameters[0] > 0 and sourmash_parameters[1] > 0:
        raise ValueError("One of the sourmash parameters should be 0")

    for p in sourmash_parameters:
        if type(p) is not int:
            raise ValueError("sourmash parameters must be both integers")

    # Handling both cases of fasta input
    raw_is_dir = os.path.isdir(raw_in)

    if raw_is_dir:  # If a directory path is provided, read all fasta files in the directory
        try:
            records = []
            rec_names = []
            for file in os.listdir(raw_in):
                rec_names.append(file.split("_reoriented.fna")[0])
                if file.endswith(".fasta") or file.endswith(".fna"):
                    records_inner = []
                    for rec in SeqIO.parse(os.path.join(raw_in, file), "fasta"):
                        records_inner.append(rec)
                    records.append(records_inner)
        except FileNotFoundError as e:
            print(f"Error: {e}. Please check the file path.")
            return 1
    else:
        try:
            records = list(SeqIO.parse(raw_in, "fasta"))
        except FileNotFoundError as e:
            print(f"Error: {e}. Please check the file path.")
            return 1

    ### Constructing minhashes for all records ###
    if not quiet: print("------- Constructing MinHashes -------")
    minhashes = []
    for rec in tqdm(records, desc="Constructing minhashes for all records", unit="seq"):
        if raw_is_dir:
            try:
                mh = sourmash.MinHash(n=sourmash_parameters[0], ksize=k, scaled=sourmash_parameters[1]) #each record gets its own minhash | scaled=1000 to limit memory usage
                for rec_inner in rec:
                    for i in range(0, len(rec_inner.seq) - k + 1):
                        kmer = str(rec_inner.seq[i:i+k])
                        mh.add_sequence(kmer, force=True)
                        if include_reverse:
                            mh.add_sequence(kmer[::-1], force=True)
                minhashes.append(mh)
            except:
                raise SystemError("Error in constructing minhashes")
        else:
            try:
                mh = sourmash.MinHash(n=sourmash_parameters[0], ksize=k, scaled=sourmash_parameters[1]) #each record gets its own minhash | scaled=1000 to limit memory usage
                for i in range(0, len(rec.seq) - k + 1):
                    kmer = str(rec.seq[i:i+k])
                    mh.add_sequence(kmer, force=True)
                    if include_reverse:
                        mh.add_sequence(kmer[::-1], force=True)
                minhashes.append(mh)
            except:
                raise SystemError("Error in constructing minhashes")

    ### Saving sketches ###
    if not quiet: print("------- Saving Sketches -------")
    if "bact" in raw_in:
        outfile_prefix = "bact"
    elif "phage" in raw_in:
        outfile_prefix = "phage"
    else:
        outfile_prefix = "out"

    for i in range(len(minhashes)):
        try:
            with open(outpath+f"{outfile_prefix}{i}_minhash.sig", "wt") as sigfile:
                if raw_is_dir:
                    sig1 = sourmash.SourmashSignature(minhashes[i], name=rec_names[i])
                else:
                    sig1 = sourmash.SourmashSignature(minhashes[i], name=records[i].id)
                sourmash.save_signatures([sig1], sigfile)
        except:
            raise SystemError(f"Error in saving sourmash sketch for: {records[i].id}")
    print("------- Process Completed -------")

In [ ]:
import os, sys 
from io_operations import presence_matrix
from paths import raw_data_path, data_prod_path

### Phage Minhash Sketch Construction ###
pk = 12
pn = 500
phage_outdir = f"PhageMinhash_n{pn}_k{pk}/"
construct_SM_sketches_dev(raw_in = raw_data_path+"phagehost_KU/phage_cleaned.fasta", 
                      k = pk, 
                      outdir = phage_outdir, 
                      quiet = False,
                      sourmash_parameters=[pn, 0])

### Bacteria Minhash Sketch Construction ###
bk = 12
bn = 500
bact_outdir = f"BactMinhash_n{bn}_k{bk}/"
construct_SM_sketches_dev(raw_in = raw_data_path+"phagehost_KU/bacteria_fasta/", 
                      k = bk, 
                      outdir = bact_outdir, 
                      quiet = False,
                      sourmash_parameters=[bn, 0])

## Novel decompisition method
Develop a new method, where i can backtrack the decomposed integers, to its original kmer sequences

Encoder Process:
1) encode the forward k-mer to bit-level integer
2) encode its reverse complement 
3) keep only the smaller of the two

Decomposition Process:
1) divide genome into kmer of size k
2) keep only every x entry, where x = n/genome_kmer_size (n = sig size)
3) save to disk

### Encoder

In [ ]:
from decompositions import KmerCodec
codec = KmerCodec()
my_kmer = "GATCGACT"
k_size = len(my_kmer)

# 1. Decompose to integer
encoded_val = codec.encode_with_revcomp(my_kmer)
print(f"Original: {my_kmer}")
print(f"Integer representation: {encoded_val}") # 8864 in decimal

# 2. Backtrack to sequence
decoded_val = codec.decode(encoded_val, k_size)
print(f"Backtracked: {decoded_val}")

### Decomposition

In [ ]:
from decompositions import Decompose
raw_data_path = "../raw_data/"
data_prod_path = "../data_prod/"

k = 12
n = 400

# The 'with' block handles the creation and deletion of tmp automatically
with Decompose(k=k, n=n, codec=codec, output_dir=data_prod_path+"encoded_sketches/", entity_type="phage", sourmash_like=True, 
               custom_dir_name=f"encode4bit_n{n}_k{k}") as decomposer:
    for line in decomposer.decompose(raw_data_path+"phagehost_KU/phage_cleaned.fasta"):
        print(line)

with Decompose(k=k, n=n, codec=codec, output_dir=data_prod_path+"encoded_sketches/", entity_type="bact", sourmash_like=True,
               custom_dir_name=f"encode4bit_n{n}_k{k}") as decomposer:
    for line in decomposer.decompose(raw_data_path+"phagehost_KU/bacteriaKU_cleaned.fasta"):
        print(line)

# Statistical Eval on down-sampled signatures